In [5]:
import os
import gc
import sys
import glob
import numpy as np
import pandas as pd
import netCDF4 as nc
from datetime import datetime, timedelta 
from matplotlib.cm import get_cmap
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib import colors
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import multiprocessing as mp

In [6]:
# To use PLUMBER2_GPP_common_utils, change directory to where it exists
os.chdir('/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_GPP_PLUMBER2')
from PLUMBER2_GPP_common_utils import *

In [7]:
# Path of PLUMBER 2 dataset
PLUMBER2_path      = "/srv/ccrc/LandAP/z5218916/data/PLUMBER2/"
PLUMBER2_flux_path = "/srv/ccrc/LandAP/z5218916/data/Fluxnet_data/Post-processed_PLUMBER2_outputs/Nc_files/Flux/"
PLUMBER2_met_path  = "/srv/ccrc/LandAP/z5218916/data/Fluxnet_data/Post-processed_PLUMBER2_outputs/Nc_files/Met/"

site_names, IGBP_types, clim_types, model_names = load_default_list()

remove_site        = get_removed_site_names()

models_calc_LAI   = ['ORC2_r6593','ORC2_r6593_CO2','ORC3_r7245_NEE','ORC3_r8120','GFDL','SDGVM','QUINCY','NoahMPv401']
model_LAI_names   = {'ORC2_r6593':'lai','ORC2_r6593_CO2':'lai','ORC3_r7245_NEE':'lai','ORC3_r8120':'lai',
                     'GFDL':'lai', 'SDGVM':'lai','QUINCY':'LAI','NoahMPv401':'LAI'} #

# Calculate remaining sites
set_site_names      = set(site_names)
set_remove_site     = set(remove_site)
remain_sites        = set_site_names - set_remove_site
remain_sites        = list(remain_sites)

In [8]:
model_colors ={0:'red', 1: 'darkorange',2:'orange',3:'gold',4:'yellowgreen',5:'green',6:'mediumseagreen',
               7:'lime',8:'aquamarine',9:'cyan',10:'dodgerblue',11:'blue',12:'darkolivegreen',
               13:'forestgreen',14:'lime',15:'gold', 16:'orange',17:'pink',18:'pink',19:'red',20:'deeppink',
               21:'mediumorchid',22: 'darkviolet',}

IGBP_colors  = set_IGBP_colors()
clim_colors  = set_clim_colors()

<h4 style="color:green;">Interannual variability NEE</h4>  

<h5 style="color:orange;">Save interannual variability NEE</h5>  

In [ ]:
def save_IAV(var_name, model_in, per_LAI=False):
    
    nsite    = len(remain_sites)
    Site_name= [""] * nsite    # Creates a list with 170 empty strings
    Variance = np.zeros(nsite)
    lat      = np.zeros(nsite)
    lon      = np.zeros(nsite)
    
    for s, site_name in enumerate(remain_sites[:]):
        
        Site_name[s] =  site_name
        
        PLUMBER2_path_site = f"/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_GPP_PLUMBER2/nc_files/{site_name}.nc"
        PLUMBER2_met_path  = "/srv/ccrc/LandAP/z5218916/data/Fluxnet_data/Post-processed_PLUMBER2_outputs/Nc_files/Flux/"
        file_path          = glob.glob(PLUMBER2_met_path+"/*"+site_name+"*.nc")

        with nc.Dataset(PLUMBER2_path_site, mode='r') as f:
            try:
                if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
                    var = pd.DataFrame(f.variables[f'{model_in}_{var_name}'][:].data*(-1), columns=[var_name])
                else:
                    var = pd.DataFrame(f.variables[f'{model_in}_{var_name}'][:].data, columns=[var_name]) 

                time       = nc.num2date(f.variables['CABLE_time'][:],f.variables['CABLE_time'].units,only_use_cftime_datetimes=False,only_use_python_datetimes=True)
                ntime      = len(time)
                year       = np.zeros(ntime)

                for i,t in enumerate(time):
                    year[i] = t.year

                var['year'] = year[:]

                var         = var.groupby(['year']).mean(numeric_only=True).reset_index()*365*24*3600

                if len(var) >=5:
                    Variance[s] = np.var(var[var_name])
                    with nc.Dataset(file_path[0], mode='r') as f_flux:
                        lat[s] = f_flux.variables['latitude'][0,0] 
                        lon[s] = f_flux.variables['longitude'][0,0]
            except:
                print(model_in, site_name, 'not exists')
                continue

    # Convert NEE to a numpy array (for easier manipulation)
    Variance = np.array(Variance)
    
    var_out              = pd.DataFrame(Site_name, columns=['site_name'])
    var_out['variance']  = Variance
    var_out['lat']       = lat
    var_out['lon']       = lon

    if per_LAI:
        var_out.to_csv(f'./txt/{var_name}_IAV_per_LAI_{model_in}.csv', index=False)
    else:
        var_out.to_csv(f'./txt/{var_name}_IAV_{model_in}.csv', index=False)

In [ ]:
# Define a function to generate each plot
def save_IAV_parallal(var_name, per_LAI=False):
    
    PLUMBER2_path_site = "/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_GPP_PLUMBER2/nc_files/AU-How.nc"
    f                  = nc.Dataset(PLUMBER2_path_site, mode='r')
    model_list         = f.variables[f'{var_name}_models'][:]
    model_list         = model_list.tolist()
    model_list.append('obs')
    f.close()

    # Create a pool of workers (28 CPUs)
    with mp.Pool(processes=28) as pool:
        # Distribute the tasks across CPUs
        pool.map(save_IAV, model_list)  # Use map for single argument functions

In [ ]:
var_name='NEE'
per_LAI=False
save_IAV_parallal(var_name, per_LAI)

<h5 style="color:orange;">Plot interannual variability</h5>  

In [ ]:
def plot_variance(var_name, model_in, per_LAI=False):

    if per_LAI:
        df = pd.read_csv(f'./txt/{var_name}_IAV_per_LAI_{model_in}.csv')
    else:
        df = pd.read_csv(f'./txt/{var_name}_IAV_{model_in}.csv')

        
    Variance = df['variance']
    lat      = df['lat']
    lon      = df['lon']
    
    # Convert NEE to a numpy array (for easier manipulation)
    Variance = np.array(Variance)

    # Define marker size based on the absolute value of NEE (scaled for better visibility)
    marker_size = np.abs(Variance) * 500  # Adjust scaling factor (100) as necessary for your data

    # Create a global map plot
    fig = plt.figure(figsize=(10, 5))
    ax  = plt.axes(projection=ccrs.PlateCarree())

    plt.rcParams['text.usetex']     = False
    plt.rcParams['font.family']     = "sans-serif"
    plt.rcParams['font.serif']      = "Helvetica"
    plt.rcParams['axes.linewidth']  = 1.5
    plt.rcParams['axes.labelsize']  = 14
    plt.rcParams['font.size']       = 14
    plt.rcParams['legend.fontsize'] = 10
    plt.rcParams['xtick.labelsize'] = 14
    plt.rcParams['ytick.labelsize'] = 14

    almost_black = '#262626'
    # change the tick colors also to the almost black
    plt.rcParams['ytick.color']     = almost_black
    plt.rcParams['xtick.color']     = almost_black

    # change the text colors also to the almost black
    plt.rcParams['text.color']      = almost_black

    # Change the default axis colors from black to a slightly lighter black,
    # and a little thinner (0.5 instead of 1)
    plt.rcParams['axes.edgecolor']  = almost_black
    plt.rcParams['axes.labelcolor'] = almost_black

    # set the box type of sequence number
    props = dict(boxstyle="round", facecolor='white', alpha=0.0, ec='white')

    # Add coastlines and other map features
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linestyle=':', color='gray')
    ax.add_feature(cfeature.LAND, facecolor='white')
    ax.add_feature(cfeature.OCEAN, edgecolor='none', facecolor="white")
    ax.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())  # [lon_min, lon_max, lat_min, lat_max]

    # Scatter plot the flux sites with colors and sizes based on NEE
    sc = ax.scatter(lon, lat, alpha=0.7, c='blue', s=30, edgecolors='none', transform=ccrs.PlateCarree()) # marker_size

    # Set title and show the plot
    plt.title(f'{model_in} {var_name} Variance')

    plt.savefig(f'./plots/Global_map_IAV_{var_name}_{model_in}.png',dpi=300)

In [ ]:
plot_variance